> **Portfolio Version:** This notebook has been prepared for portfolio purposes. Confidential data and source files have been removed.


In [ ]:
%pip install sqlalchemy pyodbc

In [ ]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy import text
import os

In [ ]:

server = r"YOUR_SERVER_NAME\YOUR_INSTANCE_NAME"
database = "Uaru_NDT_DW"
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    f"?driver={driver.replace(' ', '+')}"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

print("SQL Server connection created successfully")

In [ ]:
with engine.connect() as conn:
    current_count = conn.execute(
        text("SELECT COUNT(*) FROM dbo.fact_ndt_inspection")
    ).scalar()

print("Current SQL fact rows:", current_count)

In [ ]:
fact_ndt_inspection = pd.read_csv(
    "../01_Data/processed/fact_ndt_inspection.csv"
)

print(fact_ndt_inspection.shape)

In [ ]:
# Load dimension keys from SQL Server
dim_calendar_sql = pd.read_sql(
    "SELECT DateKey FROM dbo.dim_calendar",
    con=engine
)

dim_discipline_sql = pd.read_sql(
    "SELECT DisciplineKey FROM dbo.dim_discipline",
    con=engine
)

dim_module_sql = pd.read_sql(
    "SELECT ModuleKey FROM dbo.dim_module",
    con=engine
)

dim_method_sql = pd.read_sql(
    "SELECT MethodKey FROM dbo.dim_method",
    con=engine
)

# Build valid key sets
valid_date_keys = set(dim_calendar_sql["DateKey"].astype(int))
valid_discipline_keys = set(dim_discipline_sql["DisciplineKey"].astype(int))
valid_module_keys = set(dim_module_sql["ModuleKey"].astype(int))
valid_method_keys = set(dim_method_sql["MethodKey"].astype(int))

# Check fact foreign keys
missing_request_dates = sorted(
    set(fact_ndt_inspection["RequestDateKey"].dropna().astype(int))
    - valid_date_keys
)

missing_jdr_dates = sorted(
    set(fact_ndt_inspection["JDRDateKey"].dropna().astype(int))
    - valid_date_keys
)

missing_disciplines = sorted(
    set(fact_ndt_inspection["DisciplineKey"].dropna().astype(int))
    - valid_discipline_keys
)

missing_modules = sorted(
    set(fact_ndt_inspection["ModuleKey"].dropna().astype(int))
    - valid_module_keys
)

missing_methods = sorted(
    set(fact_ndt_inspection["MethodKey"].dropna().astype(int))
    - valid_method_keys
)

print("Missing RequestDateKeys:", missing_request_dates)
print("Missing JDRDateKeys:", missing_jdr_dates)
print("Missing DisciplineKeys:", missing_disciplines)
print("Missing ModuleKeys:", missing_modules)
print("Missing MethodKeys:", missing_methods)

In [ ]:
from sqlalchemy import text

with engine.begin() as conn:
    # Remove current fact rows but preserve the table,
    # primary key, foreign keys, and data types.
    conn.execute(text("DELETE FROM dbo.fact_ndt_inspection"))

print("Existing SQL fact rows deleted.")

In [ ]:
with engine.connect() as conn:
    empty_count = conn.execute(
        text("SELECT COUNT(*) FROM dbo.fact_ndt_inspection")
    ).scalar()

print("SQL fact rows after delete:", empty_count)

In [ ]:
fact_ndt_inspection.to_sql(
    name="fact_ndt_inspection",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Corrected fact data loaded successfully.")

In [ ]:
with engine.connect() as conn:
    final_count = conn.execute(
        text("SELECT COUNT(*) FROM dbo.fact_ndt_inspection")
    ).scalar()

print("Final SQL fact rows:", final_count)

In [ ]:
validation_queries = {
    "Fact row count": """
        SELECT COUNT(*)
        FROM dbo.fact_ndt_inspection
    """,

    "Duplicate InspectionID": """
        SELECT COUNT(*)
        FROM (
            SELECT InspectionID
            FROM dbo.fact_ndt_inspection
            GROUP BY InspectionID
            HAVING COUNT(*) > 1
        ) d
    """,

    "Missing RequestDate FK": """
        SELECT COUNT(*)
        FROM dbo.fact_ndt_inspection f
        LEFT JOIN dbo.dim_calendar c
            ON f.RequestDateKey = c.DateKey
        WHERE c.DateKey IS NULL
    """,

    "Missing JDRDate FK": """
        SELECT COUNT(*)
        FROM dbo.fact_ndt_inspection f
        LEFT JOIN dbo.dim_calendar c
            ON f.JDRDateKey = c.DateKey
        WHERE c.DateKey IS NULL
    """
}

with engine.connect() as conn:
    for check_name, query in validation_queries.items():
        result = conn.execute(text(query)).scalar()
        print(f"{check_name}: {result}")

In [ ]:
# dim_method = pd.read_csv("../Output/dim_method.csv")

# dim_method.head()

In [ ]:
# dim_method.to_sql(
#     name="dim_method",
#     con=engine,
#     if_exists="replace",
#     index=False
# )

In [ ]:
# with engine.connect() as conn:
#     result = conn.execute(text("SELECT COUNT(*) FROM dim_method"))
#     print(result.scalar())

In [ ]:
# dim_discipline = pd.read_csv("../Output/dim_discipline.csv")

# dim_discipline.head()

In [ ]:
# dim_discipline.to_sql(
#     name="dim_discipline",
#     con=engine,
#     if_exists="replace",
#     index=False
# )


In [ ]:
# with engine.connect() as conn:
#     result = conn.execute(text("SELECT COUNT(*) FROM dim_discipline"))
#     print(result.scalar())

In [ ]:
# dim_module = pd.read_csv("../Output/dim_module.csv")

# dim_module.head()

In [ ]:
# dim_module.to_sql(
#     name="dim_module",
#     con=engine,
#     if_exists="replace",
#     index=False
# )

In [ ]:
# with engine.connect() as conn:
#     result = conn.execute(text("SELECT COUNT(*) FROM dim_module"))
#     print(result.scalar())

In [ ]:
# dim_calendar = pd.read_csv("../Output/dim_calendar.csv")

# dim_calendar.head()

In [ ]:
# dim_calendar.to_sql(
#     name="dim_calendar",
#     con=engine,
#     if_exists="replace",
#     index=False
# )

In [ ]:
# with engine.connect() as conn:
#     result = conn.execute(text("SELECT COUNT(*) FROM dim_calendar"))
#     print(result.scalar())